# NeuroScope - ablation and baseline (Kaggle, GPU T4 x2, Internet on)
Attach two inputs: the BraTS 2020 dataset (`awsaf49/brats20-dataset-training-validation`) and the output of the earlier run
(the notebook that contains `neuroscope_artifacts.zip`). Then **Save & Run All** (about 55 min).

1. **Ablation** - same U-Net trained *without* modality dropout, evaluated on all 15 sequence subsets.
2. **Baseline** - the trained model evaluated with a single deterministic pass (MC=1) vs Monte Carlo dropout (MC=8): does MC dropout improve failure detection?

In [ ]:
# 1) Code
REPO_URL = "https://github.com/neetii-git/neuroscope.git"
import os
if not os.path.isdir("/kaggle/working/neuroscope"):
    !git clone -q {REPO_URL} /kaggle/working/neuroscope
%cd /kaggle/working/neuroscope
!git log --oneline -1
!pip install -q nibabel pyyaml 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Preprocess (same seed 42 -> identical patient-level split to the main run)
import glob, os, json
hits = glob.glob("/kaggle/input/**/MICCAI_BraTS2020_TrainingData", recursive=True)
assert hits, "Attach the BraTS2020 dataset under Input."
RAW = hits[0]; CACHE = "/kaggle/working/cache"
!python scripts/preprocess.py --raw {RAW} --cache {CACHE}
s = json.load(open(f"{CACHE}/split.json")); print({k: len(v) for k, v in s.items() if isinstance(v, list)})

In [ ]:
# 3) Restore the main (modality-dropout) model from the earlier run's artifacts
import glob, zipfile
z = glob.glob("/kaggle/input/**/neuroscope_artifacts.zip", recursive=True)
assert z, "Attach the earlier notebook's output (contains neuroscope_artifacts.zip) under Input."
with zipfile.ZipFile(z[0]) as f: f.extractall("/kaggle/working")
assert os.path.exists("/kaggle/working/runs/base/best.pt")
print(os.listdir("/kaggle/working/runs/base"))

In [ ]:
# 4) ABLATION: train without modality dropout (~35 min), then evaluate on the same test set and all 15 subsets
!python scripts/train.py --cache /kaggle/working/cache --out /kaggle/working/runs/no_moddrop --epochs 20 --no-modality-dropout
!python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/no_moddrop/best.pt --out /kaggle/working/runs/no_moddrop/results --skip-corruptions --no-hd95

In [ ]:
# 5) BASELINE: main model, single deterministic pass (MC=1) - no Monte Carlo dropout
!python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/base/best.pt --out /kaggle/working/runs/base/results_mc1 --mc 1 --skip-subsets --skip-corruptions --no-hd95

In [ ]:
# 6) Comparison tables
import pandas as pd, json
R = "/kaggle/working/runs"
a = pd.read_csv(f"{R}/base/results/subsets.csv").set_index("subset")
b = pd.read_csv(f"{R}/no_moddrop/results/subsets.csv").set_index("subset")
cmp = pd.DataFrame({"n_mod": a["n_modalities"], "WT_with": a["dice_WT"], "WT_without": b["dice_WT"],
                    "TC_with": a["dice_TC"], "TC_without": b["dice_TC"], "ET_with": a["dice_ET"], "ET_without": b["dice_ET"]}).round(3)
display(cmp.sort_values("n_mod"))
print(cmp.groupby("n_mod").mean().round(3))
sm = {k: json.load(open(f"{R}/{k}/summary.json")) for k in ["base/results", "no_moddrop/results", "base/results_mc1"]}
rows = []
for k, v in sm.items():
    rows.append({"run": k, "Dice WT": v["dice_WT"]["mean"], "Dice TC": v["dice_TC"]["mean"], "Dice ET": v["dice_ET"]["mean"],
                 "failure AUROC": v["reliability"]["failure_auroc"], "spearman rho": v["reliability"]["spearman_rho"], "AURC": v["reliability"]["aurc"],
                 "flag rate": v["review_flag"]["flag_rate"], "flag sens": v["review_flag"]["flag_sensitivity"], "Dice accepted": v["review_flag"]["dice_accepted"]})
display(pd.DataFrame(rows).round(3))

In [ ]:
# 7) Package only the small result files (no checkpoints)
!mkdir -p /kaggle/working/ablation && cp -r /kaggle/working/runs/no_moddrop/results /kaggle/working/ablation/no_moddrop_results && cp /kaggle/working/runs/no_moddrop/history.csv /kaggle/working/ablation/no_moddrop_history.csv && cp -r /kaggle/working/runs/base/results_mc1 /kaggle/working/ablation/base_mc1_results
!cd /kaggle/working && zip -q -r ablation_artifacts.zip ablation && ls -lh ablation_artifacts.zip